# Plot Experiment Training Curves

This notebook reads local cluster outputs from `runs/*/seed_*` and plots return/success curves averaged across seeds.

Use `CURVE_SOURCE = "mean"` for smoother W&B-style training curves from `scalars.csv`, or `CURVE_SOURCE = "raw"` for raw per-episode curves from `episodes.csv`.


## 1. Settings


In [1]:
from pathlib import Path

RUNS_DIR = Path("runs")
FIG_DIR = Path("figures/training_curves")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# "mean" -> scalars.csv: mean_return_100 and success_rate_100
# "raw"  -> episodes.csv: raw episode return and raw episode success
CURVE_SOURCE = "mean"

# Show mean +/- standard error across seeds as a fill band.
SHOW_SE = False

# Curves are averaged across seeds on a common step grid.
N_BINS = 250
MIN_SEEDS_TO_DRAW = 1

# E2 has three beta settings. Plot-2 shows all; Plot-4 uses this beta by default
# so the all-method comparison remains readable and matches E3's fixed beta.
COMPARISON_BETA = 0.75

# The cross-experiment comparison is for the four environments shared by E2/E3.
COMPARISON_ENVS = ["taxi", "doorkey6x6", "unlockpickup", "redbluedoors6x6"]

SAVE_DPI = 300


## 2. Imports And Style


In [2]:
import json
import os
import re
from dataclasses import dataclass

os.environ.setdefault("MPLCONFIGDIR", str((Path(".matplotlib_cache")).resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

if CURVE_SOURCE not in {"mean", "raw"}:
    raise ValueError('CURVE_SOURCE must be "mean" or "raw"')

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 18,
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.linewidth": 1.8,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 14,
    "figure.titlesize": 24,
})

ENV_LABELS = {
    "doorkey5x5": "DoorKey 5x5",
    "doorkey6x6": "DoorKey 6x6",
    "lavagap": "LavaGap",
    "unlock": "Unlock",
    "unlockpickup": "UnlockPickup",
    "redbluedoors6x6": "RedBlue 6x6",
    "taxi": "Taxi",
}

ENV_ORDER_E1 = ["doorkey5x5", "doorkey6x6", "lavagap", "unlock", "unlockpickup", "redbluedoors6x6", "taxi"]
ENV_ORDER_E23 = ["taxi", "doorkey6x6", "unlockpickup", "redbluedoors6x6"]

COLORS = {
    "PPO": "#0072B2",              # blue
    "PPO-CF": "#D55E00",           # vermillion
    "Queried b=0.25": "#009E73",   # green
    "Queried b=0.75": "#CC79A7",   # magenta
    "Queried b=1.5": "#56B4E9",    # sky blue
    "Distill b=0.25": "#E69F00",   # orange
    "Distill b=0.75": "#F0E442",   # yellow
    "Distill b=1.5": "#000000",    # black
    "Uniform": "#7F3C8D",          # purple
    "Uncertainty": "#11A579",      # teal
    "Active": "#E73F74",           # rose
}

LINESTYLES = {label: "-" for label in COLORS}


def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)
    ax.tick_params(width=1.5, length=5)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")


def savefig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, dpi=SAVE_DPI, bbox_inches="tight")
    print("saved", path)


Matplotlib is building the font cache; this may take a moment.


## 3. Load Training Data


In [3]:
def parse_beta(text):
    match = re.search(r"_b(\d+)p(\d+)", text)
    if match:
        return float(f"{match.group(1)}.{match.group(2)}")
    return np.nan


def parse_run_dir(name):
    # Experiment 1: exp1_<env>_<gae|cf>
    m = re.match(r"^exp1_(.+)_(gae|cf)$", name)
    if m:
        env_raw, arm = m.groups()
        env = {
            "dk5": "doorkey5x5",
            "dk6": "doorkey6x6",
            "rbd6": "redbluedoors6x6",
        }.get(env_raw, env_raw)
        return {
            "experiment": "Experiment 1",
            "env": env,
            "method": "PPO" if arm == "gae" else "PPO-CF",
            "arm": arm,
            "beta": np.nan,
            "query_strategy": "",
        }

    # Experiment 2: e2_<env>_<queried|distill>_b...
    m = re.match(r"^e2_(.+)_(queried|distill)_b\d+p\d+$", name)
    if m:
        env_raw, arm = m.groups()
        env = {"dk6": "doorkey6x6", "rbd6": "redbluedoors6x6"}.get(env_raw, env_raw)
        beta = parse_beta(name)
        label = f"{'Queried' if arm == 'queried' else 'Distill'} b={beta:g}"
        return {
            "experiment": "Experiment 2",
            "env": env,
            "method": label,
            "arm": arm,
            "beta": beta,
            "query_strategy": "",
        }

    # Experiment 3: e3_<env>_<uniform|uncertainty|active>_b...
    m = re.match(r"^e3_(.+)_(uniform|uncertainty|active)_b\d+p\d+$", name)
    if m:
        env_raw, arm = m.groups()
        env = {"dk6": "doorkey6x6", "rbd6": "redbluedoors6x6"}.get(env_raw, env_raw)
        label = {"uniform": "Uniform", "uncertainty": "Uncertainty", "active": "Active"}[arm]
        return {
            "experiment": "Experiment 3",
            "env": env,
            "method": label,
            "arm": arm,
            "beta": parse_beta(name),
            "query_strategy": arm,
        }
    return None


def load_seed_curve(seed_dir):
    if CURVE_SOURCE == "mean":
        path = seed_dir / "scalars.csv"
        if not path.exists():
            return None
        df = pd.read_csv(path)
        needed = {"global_step", "mean_return_100", "success_rate_100"}
        missing = needed - set(df.columns)
        if missing:
            print("skip", path, "missing", missing)
            return None
        out = df[["global_step", "mean_return_100", "success_rate_100"]].copy()
        out = out.rename(columns={"mean_return_100": "return", "success_rate_100": "success"})
        return out

    path = seed_dir / "episodes.csv"
    if not path.exists():
        return None
    df = pd.read_csv(path)
    needed = {"global_step", "return", "success"}
    missing = needed - set(df.columns)
    if missing:
        print("skip", path, "missing", missing)
        return None
    out = df[["global_step", "return", "success"]].copy()
    out["success"] = out["success"].astype(bool).astype(float)
    return out


rows = []
for run_dir in sorted(RUNS_DIR.iterdir() if RUNS_DIR.exists() else []):
    if not run_dir.is_dir():
        continue
    meta = parse_run_dir(run_dir.name)
    if meta is None:
        continue
    for seed_dir in sorted(run_dir.glob("seed_*")):
        seed_match = re.match(r"seed_(\d+)$", seed_dir.name)
        seed = int(seed_match.group(1)) if seed_match else np.nan
        curve = load_seed_curve(seed_dir)
        if curve is None or curve.empty:
            continue
        curve["seed"] = seed
        curve["run_name"] = run_dir.name
        for k, v in meta.items():
            curve[k] = v
        rows.append(curve)

data = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
print("source:", CURVE_SOURCE)
print("rows:", len(data))
print("runs:", data["run_name"].nunique() if len(data) else 0)
display(
    data.groupby(["experiment", "env", "method"], dropna=False)["seed"]
    .nunique()
    .reset_index(name="n_seeds")
    .sort_values(["experiment", "env", "method"])
)


source: mean
rows: 0
runs: 0


KeyError: 'experiment'

## 4. Aggregate Curves Across Seeds


In [ ]:
def bin_one_seed(df, metric, n_bins=N_BINS):
    df = df.sort_values("global_step")
    if df.empty:
        return pd.DataFrame(columns=["grid_step", "global_step", metric])
    max_step = float(df["global_step"].max())
    if max_step <= 0:
        return pd.DataFrame(columns=["grid_step", "global_step", metric])
    bins = np.linspace(0.0, max_step, n_bins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])
    out = df.copy()
    out["bin"] = pd.cut(out["global_step"], bins=bins, include_lowest=True, labels=False)
    grouped = out.dropna(subset=["bin"]).groupby("bin", as_index=False).agg(value=(metric, "mean"))
    grouped["bin"] = grouped["bin"].astype(int)
    grouped["grid_step"] = centers[grouped["bin"].to_numpy()]
    grouped = grouped.rename(columns={"value": metric})
    grouped["global_step"] = grouped["grid_step"]
    return grouped[["grid_step", "global_step", metric]]


def aggregate_curves(source_data, group_cols, metric):
    pieces = []
    if source_data.empty:
        return pd.DataFrame(columns=[*group_cols, "global_step", "mean", "sem", "n_seeds"])
    for keys, group in source_data.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        seed_curves = []
        for seed, sdf in group.groupby("seed"):
            c = bin_one_seed(sdf, metric)
            if len(c):
                c = c.rename(columns={metric: "value"})
                c["seed"] = seed
                seed_curves.append(c)
        if not seed_curves:
            continue
        seed_df = pd.concat(seed_curves, ignore_index=True)
        agg = seed_df.groupby("global_step", as_index=False).agg(
            mean=("value", "mean"),
            sem=("value", lambda x: float(x.std(ddof=1) / np.sqrt(len(x))) if len(x) > 1 else 0.0),
            n_seeds=("seed", "nunique"),
        )
        for col, val in zip(group_cols, keys):
            agg[col] = val
        pieces.append(agg[[*group_cols, "global_step", "mean", "sem", "n_seeds"]])
    return pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame(columns=[*group_cols, "global_step", "mean", "sem", "n_seeds"])


GROUP_COLS = ["experiment", "env", "method", "arm", "beta", "query_strategy"]
return_curves = aggregate_curves(data, GROUP_COLS, "return")
success_curves = aggregate_curves(data, GROUP_COLS, "success")
print("return curve rows:", len(return_curves))
print("success curve rows:", len(success_curves))


## 5. Plot Helpers


In [ ]:
def subset_curve(curves, experiment=None, envs=None, methods=None, beta=None):
    out = curves.copy()
    if experiment is not None:
        out = out[out["experiment"].eq(experiment)]
    if envs is not None:
        out = out[out["env"].isin(envs)]
    if methods is not None:
        out = out[out["method"].isin(methods)]
    if beta is not None:
        out = out[(out["beta"].isna()) | np.isclose(out["beta"].astype(float), float(beta), equal_nan=False)]
    return out


def draw_curve(ax, df, label):
    df = df.sort_values("global_step")
    if df.empty or df["n_seeds"].max() < MIN_SEEDS_TO_DRAW:
        return
    color = COLORS.get(label, None)
    linestyle = LINESTYLES.get(label, "-")
    x = df["global_step"].to_numpy(dtype=float)
    y = df["mean"].to_numpy(dtype=float)
    se = df["sem"].fillna(0.0).to_numpy(dtype=float)
    ax.plot(x, y, label=label, color=color, linestyle=linestyle, linewidth=2.6)
    if SHOW_SE:
        ax.fill_between(x, y - se, y + se, color=color, alpha=0.16, linewidth=0)


def plot_experiment_grid(experiment, env_order, filename_prefix):
    ret = subset_curve(return_curves, experiment=experiment, envs=env_order)
    suc = subset_curve(success_curves, experiment=experiment, envs=env_order)
    methods = sorted(ret["method"].dropna().unique().tolist())

    n_env = len(env_order)
    fig, axes = plt.subplots(2, n_env, figsize=(5.0 * n_env, 8.0), sharex=False)
    if n_env == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    for col, env in enumerate(env_order):
        for row, (metric_name, curves, ylabel) in enumerate([
            ("return", ret, "Return"),
            ("success", suc, "Success"),
        ]):
            ax = axes[row, col]
            env_curves = curves[curves["env"].eq(env)]
            for method in methods:
                draw_curve(ax, env_curves[env_curves["method"].eq(method)], method)
            ax.set_title(ENV_LABELS.get(env, env), pad=12)
            ax.set_xlabel("Step")
            if col == 0:
                ax.set_ylabel(ylabel)
            clean_axes(ax)
            if metric_name == "success":
                ax.set_ylim(-0.03, 1.03)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=min(len(labels), 6), frameon=False, bbox_to_anchor=(0.5, 1.03))
    source_label = "Mean Scalars" if CURVE_SOURCE == "mean" else "Raw Episodes"
    se_label = " with SE" if SHOW_SE else ""
    fig.suptitle(f"{experiment}: {source_label} Training Curves{se_label}", y=1.10)
    fig.tight_layout()
    savefig(fig, f"{filename_prefix}_{CURVE_SOURCE}_return_success.png")
    return fig


def plot_comparison_grid(env_order=COMPARISON_ENVS, beta=COMPARISON_BETA):
    e1_methods = ["PPO", "PPO-CF"]
    e2_methods = [f"Queried b={beta:g}", f"Distill b={beta:g}"]
    e3_methods = ["Uniform", "Uncertainty", "Active"]
    methods = e1_methods + e2_methods + e3_methods

    ret = pd.concat([
        subset_curve(return_curves, experiment="Experiment 1", envs=env_order, methods=e1_methods),
        subset_curve(return_curves, experiment="Experiment 2", envs=env_order, methods=e2_methods, beta=beta),
        subset_curve(return_curves, experiment="Experiment 3", envs=env_order, methods=e3_methods, beta=beta),
    ], ignore_index=True)
    suc = pd.concat([
        subset_curve(success_curves, experiment="Experiment 1", envs=env_order, methods=e1_methods),
        subset_curve(success_curves, experiment="Experiment 2", envs=env_order, methods=e2_methods, beta=beta),
        subset_curve(success_curves, experiment="Experiment 3", envs=env_order, methods=e3_methods, beta=beta),
    ], ignore_index=True)

    n_env = len(env_order)
    fig, axes = plt.subplots(2, n_env, figsize=(5.2 * n_env, 8.2), sharex=False)
    if n_env == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    for col, env in enumerate(env_order):
        for row, (metric_name, curves, ylabel) in enumerate([
            ("return", ret, "Return"),
            ("success", suc, "Success"),
        ]):
            ax = axes[row, col]
            env_curves = curves[curves["env"].eq(env)]
            for method in methods:
                draw_curve(ax, env_curves[env_curves["method"].eq(method)], method)
            ax.set_title(ENV_LABELS.get(env, env), pad=12)
            ax.set_xlabel("Step")
            if col == 0:
                ax.set_ylabel(ylabel)
            clean_axes(ax)
            if metric_name == "success":
                ax.set_ylim(-0.03, 1.03)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=min(len(labels), 7), frameon=False, bbox_to_anchor=(0.5, 1.05))
    source_label = "Mean Scalars" if CURVE_SOURCE == "mean" else "Raw Episodes"
    se_label = " with SE" if SHOW_SE else ""
    fig.suptitle(f"Cross-Experiment Comparison: {source_label}{se_label} (E2/E3 beta={beta:g})", y=1.12)
    fig.tight_layout()
    savefig(fig, f"plot4_all_experiments_comparison_b{str(beta).replace('.', 'p')}_{CURVE_SOURCE}.png")
    return fig


def plot_ppo_vs_active_uncertainty(env_order=COMPARISON_ENVS, beta=COMPARISON_BETA):
    methods = ["PPO", "PPO-CF", "Uncertainty", "Active"]

    ret = pd.concat([
        subset_curve(return_curves, experiment="Experiment 1", envs=env_order, methods=["PPO", "PPO-CF"]),
        subset_curve(return_curves, experiment="Experiment 3", envs=env_order, methods=["Uncertainty", "Active"], beta=beta),
    ], ignore_index=True)
    suc = pd.concat([
        subset_curve(success_curves, experiment="Experiment 1", envs=env_order, methods=["PPO", "PPO-CF"]),
        subset_curve(success_curves, experiment="Experiment 3", envs=env_order, methods=["Uncertainty", "Active"], beta=beta),
    ], ignore_index=True)

    n_env = len(env_order)
    fig, axes = plt.subplots(2, n_env, figsize=(5.2 * n_env, 8.2), sharex=False)
    if n_env == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    for col, env in enumerate(env_order):
        for row, (metric_name, curves, ylabel) in enumerate([
            ("return", ret, "Return"),
            ("success", suc, "Success"),
        ]):
            ax = axes[row, col]
            env_curves = curves[curves["env"].eq(env)]
            for method in methods:
                draw_curve(ax, env_curves[env_curves["method"].eq(method)], method)
            ax.set_title(ENV_LABELS.get(env, env), pad=12)
            ax.set_xlabel("Step")
            if col == 0:
                ax.set_ylabel(ylabel)
            clean_axes(ax)
            if metric_name == "success":
                ax.set_ylim(-0.03, 1.03)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=min(len(labels), 4), frameon=False, bbox_to_anchor=(0.5, 1.05))
    source_label = "Mean Scalars" if CURVE_SOURCE == "mean" else "Raw Episodes"
    se_label = " with SE" if SHOW_SE else ""
    fig.suptitle(f"PPO / PPO-CF vs E3 Uncertainty / Active: {source_label}{se_label}", y=1.12)
    fig.tight_layout()
    savefig(fig, f"plot5_ppo_ppocf_uncertainty_active_{CURVE_SOURCE}.png")
    return fig


## 6. Plot 1: Experiment 1


In [ ]:
fig_e1 = plot_experiment_grid("Experiment 1", ENV_ORDER_E1, "plot1_experiment_1")


## 7. Plot 2: Experiment 2


In [ ]:
fig_e2 = plot_experiment_grid("Experiment 2", ENV_ORDER_E23, "plot2_experiment_2")


## 8. Plot 3: Experiment 3


In [ ]:
fig_e3 = plot_experiment_grid("Experiment 3", ENV_ORDER_E23, "plot3_experiment_3")


## 9. Plot 4: Cross-Experiment Comparison


In [ ]:
fig_cmp = plot_comparison_grid(COMPARISON_ENVS, COMPARISON_BETA)


## 10. Plot 5: PPO/PPO-CF vs Uncertainty/Active


In [ ]:
fig_ppo_active = plot_ppo_vs_active_uncertainty(COMPARISON_ENVS, COMPARISON_BETA)


## 11. Optional: Save Aggregated Curve CSVs


In [ ]:
curve_dir = FIG_DIR / "curve_csv"
curve_dir.mkdir(parents=True, exist_ok=True)
return_curves.to_csv(curve_dir / f"return_curves_{CURVE_SOURCE}.csv", index=False)
success_curves.to_csv(curve_dir / f"success_curves_{CURVE_SOURCE}.csv", index=False)
data.groupby(["experiment", "env", "method", "seed"], dropna=False).size().reset_index(name="n_rows").to_csv(curve_dir / f"curve_source_counts_{CURVE_SOURCE}.csv", index=False)
print("saved curve CSVs under", curve_dir)
